In [2]:
import micropip
await micropip.install('seaborn')

In [3]:
import pandas as pd 
import numpy as np   
import matplotlib  
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import pickle
import os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION  –  change DATA_FILE to your CSV path
# ─────────────────────────────────────────────────────────────────────────────
DATA_FILE    = 'HospitalLengthofStay_SelectedFeatures_ESC113_10k.csv'
OUTPUT_DIR   = 'outputs'
TARGET_RAW   = 'hospital_los'   # original column name (minutes)
TARGET       = 'los_days'       # derived column (days)
RANDOM_STATE = 42
TEST_SIZE    = 0.20
OUTLIER_PCT  = 0.99             # drop records above this LOS percentile
MISSING_THRESH = 0.70           # drop features with > this fraction missing

os.makedirs(OUTPUT_DIR, exist_ok=True)

# colour palette
C1, C2, C3, C4 = '#2563EB', '#16A34A', '#DC2626', '#D97706'

# ═════════════════════════════════════════════════════════════════════════════
# 1. LOAD & EXPLORE
# ═════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("STEP 1 — LOAD & EXPLORE")
print("=" * 60)

df = pd.read_csv(DATA_FILE)
print(f"Raw shape          : {df.shape}")
print(f"\nTarget column stats (minutes):")
print(df[TARGET_RAW].describe())

print(f"\nMissing values (% per column, top 15):")
miss_pct = df.isnull().mean() * 100
print(miss_pct.sort_values(ascending=False).head(15).round(1))

print(f"\nData types:")
print(df.dtypes.value_counts())

# ═════════════════════════════════════════════════════════════════════════════
# 2. PREPROCESS
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 2 — PREPROCESS")
print("=" * 60)

# 2a  Convert minutes → days
df[TARGET] = df[TARGET_RAW] / 1440
df.drop(columns=[TARGET_RAW], inplace=True)
print(f"Converted {TARGET_RAW} → {TARGET} (÷ 1440)")

# 2b  Remove extreme outliers
p_cutoff = df[TARGET].quantile(OUTLIER_PCT)
n_before = len(df)
df = df[df[TARGET] <= p_cutoff].copy()
print(f"Outlier removal  : dropped {n_before - len(df):,} rows "
      f"(LOS > {p_cutoff:.1f} days, >{int(OUTLIER_PCT*100)}th pct) → {len(df):,} remain")

# 2c  Drop high-missing columns
miss = df.isnull().mean()
drop_cols = miss[miss > MISSING_THRESH].index.tolist()
df.drop(columns=drop_cols, inplace=True)
print(f"Dropped {len(drop_cols)} columns with >{int(MISSING_THRESH*100)}% missing: {drop_cols}")

# 2d  Median imputation for remaining NaN
feature_cols = [c for c in df.columns if c != TARGET]
medians = df[feature_cols].median()
df[feature_cols] = df[feature_cols].fillna(medians)
print(f"Remaining NaN after imputation : {df.isnull().sum().sum()}")
print(f"Final clean shape : {df.shape}")

# 2e  Train / test split
X = df[feature_cols]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"Train : {len(X_train):,} rows  |  Test : {len(X_test):,} rows")

# 2f  Scale for linear models
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ═════════════════════════════════════════════════════════════════════════════
# 3. TRAIN MODELS
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 3 — TRAIN MODELS")
print("=" * 60)

models = {
    'Linear Regression' : LinearRegression(),
    'Ridge Regression'  : Ridge(alpha=10.0),
    'Random Forest'     : RandomForestRegressor(
                              n_estimators=200, max_depth=12,
                              min_samples_leaf=5, n_jobs=-1,
                              random_state=RANDOM_STATE),
    'Gradient Boosting' : GradientBoostingRegressor(
                              n_estimators=200, learning_rate=0.05,
                              max_depth=5, random_state=RANDOM_STATE),
}

# linear models use scaled data; tree models use raw values
linear_models = {'Linear Regression', 'Ridge Regression'}

results = {}
for name, model in models.items():
    Xtr = X_train_sc if name in linear_models else X_train.values
    Xte = X_test_sc  if name in linear_models else X_test.values
    model.fit(Xtr, y_train)
    preds = np.maximum(model.predict(Xte), 0)   # clip negatives to 0
    mae   = mean_absolute_error(y_test, preds)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    r2    = r2_score(y_test, preds)
    results[name] = {'model': model, 'preds': preds,
                     'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(f"  {name:<22}  R²={r2:.3f}  MAE={mae:.2f} days  RMSE={rmse:.2f} days")

# ═════════════════════════════════════════════════════════════════════════════
# 4. EVALUATE — pick best model
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 4 — EVALUATION SUMMARY")
print("=" * 60)

best_name = max(results, key=lambda n: results[n]['R2'])
best      = results[best_name]
print(f"\n🏆  Best model : {best_name}")
print(f"    R²         : {best['R2']:.4f}")
print(f"    MAE        : {best['MAE']:.4f} days")
print(f"    RMSE       : {best['RMSE']:.4f} days")

metrics_df = pd.DataFrame([
    {'Model': n, 'R2': round(results[n]['R2'], 4),
     'MAE_days': round(results[n]['MAE'], 3),
     'RMSE_days': round(results[n]['RMSE'], 3)}
    for n in results
]).sort_values('R2', ascending=False)

print("\nFull comparison:")
print(metrics_df.to_string(index=False))
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'model_metrics.csv'), index=False)

# Save best model
with open(os.path.join(OUTPUT_DIR, 'best_model.pkl'), 'wb') as f:
    pickle.dump({'name': best_name, 'model': best['model'],
                 'scaler': scaler, 'features': list(X.columns)}, f)
print(f"\nBest model saved → {OUTPUT_DIR}/best_model.pkl")

# ═════════════════════════════════════════════════════════════════════════════
# 5. VISUALISATIONS
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 5 — VISUALISATIONS")
print("=" * 60)

# ── Figure 1 : EDA ────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 16))
fig.patch.set_facecolor('#F8FAFC')
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.35)

# 1a  LOS histogram
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor('white')
vals = df[TARGET].values
ax1.hist(vals, bins=80, color=C1, edgecolor='white', linewidth=0.4, alpha=0.9)
ax1.axvline(np.median(vals), color=C3, lw=2, ls='--',
            label=f'Median = {np.median(vals):.1f} d')
ax1.axvline(np.mean(vals),   color=C4, lw=2, ls='--',
            label=f'Mean = {np.mean(vals):.1f} d')
ax1.set_xlabel('Length of Stay (days)', fontsize=11)
ax1.set_ylabel('Count', fontsize=11)
ax1.set_title('Target Distribution — LOS (days)', fontsize=13, fontweight='bold', pad=10)
ax1.legend(fontsize=10)
ax1.spines[['top', 'right']].set_visible(False)

# 1b  Missing-value bar chart (original dataset)
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor('white')
miss_orig = pd.read_csv(DATA_FILE).isnull().mean() * 100
miss_orig = miss_orig[miss_orig > 0].sort_values(ascending=True)
bar_colors = [C3 if v > 70 else C1 for v in miss_orig.values]
ax2.barh(range(len(miss_orig)), miss_orig.values, color=bar_colors, edgecolor='white')
ax2.axvline(70, color=C3, lw=1.5, ls='--', label='70 % threshold (dropped)')
ax2.set_yticks(range(len(miss_orig)))
ax2.set_yticklabels([c[:28] for c in miss_orig.index], fontsize=7)
ax2.set_xlabel('Missing %', fontsize=11)
ax2.set_title('Missing Values by Feature', fontsize=13, fontweight='bold', pad=10)
ax2.legend(fontsize=9)
ax2.spines[['top', 'right']].set_visible(False)

# 1c  Age vs LOS scatter coloured by ASA score
ax3 = fig.add_subplot(gs[1, 0])
ax3.set_facecolor('white')
sample = df.sample(2000, random_state=1)
sc = ax3.scatter(sample['age'], sample[TARGET], alpha=0.35, s=15,
                 c=sample['asa'], cmap='RdYlGn_r', edgecolors='none')
plt.colorbar(sc, ax=ax3, label='ASA Score')
ax3.set_xlabel('Age (years)', fontsize=11)
ax3.set_ylabel('LOS (days)', fontsize=11)
ax3.set_title('Age vs LOS — coloured by ASA Score', fontsize=13, fontweight='bold', pad=10)
ax3.spines[['top', 'right']].set_visible(False)

# 1d  Correlation heatmap (top features vs target)
ax4 = fig.add_subplot(gs[1, 1])
ax4.set_facecolor('white')
top_feats = (df.corr()[TARGET].drop(TARGET)
               .abs().sort_values(ascending=False).head(14).index.tolist())
corr_mat  = df[top_feats + [TARGET]].corr()
mask      = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, ax=ax4, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 6.5},
            linewidths=0.4, cbar_kws={'shrink': 0.8})
ax4.set_title('Correlation Heatmap (Top Features)', fontsize=13, fontweight='bold', pad=10)
ax4.tick_params(axis='x', rotation=45, labelsize=7)
ax4.tick_params(axis='y', rotation=0,  labelsize=7)

fig.suptitle('Hospital Length of Stay — Exploratory Data Analysis',
             fontsize=16, fontweight='bold', y=1.01, color='#1E293B')
path1 = os.path.join(OUTPUT_DIR, 'fig1_eda.png')
plt.savefig(path1, dpi=150, bbox_inches='tight', facecolor='#F8FAFC')
plt.close()
print(f"  Saved → {path1}")

# ── Figure 2 : Model comparison + actual vs predicted + residuals ─────────────
model_names = list(results.keys())
r2_vals     = [results[n]['R2']   for n in model_names]
mae_vals    = [results[n]['MAE']  for n in model_names]
rmse_vals   = [results[n]['RMSE'] for n in model_names]
bar_colors  = [C2 if n == best_name else C1 for n in model_names]

fig2 = plt.figure(figsize=(20, 14))
fig2.patch.set_facecolor('#F8FAFC')
gs2  = gridspec.GridSpec(2, 3, figure=fig2, hspace=0.45, wspace=0.38)

# R² bar
ax = fig2.add_subplot(gs2[0, 0])
ax.set_facecolor('white')
bars = ax.bar(model_names, r2_vals, color=bar_colors, edgecolor='white', width=0.55)
for b, v in zip(bars, r2_vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.005,
            f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0, max(r2_vals) * 1.15)
ax.set_ylabel('R² Score', fontsize=11)
ax.set_title('R² — All Models', fontsize=13, fontweight='bold', pad=10)
ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

# MAE bar
ax = fig2.add_subplot(gs2[0, 1])
ax.set_facecolor('white')
bars = ax.bar(model_names, mae_vals, color=bar_colors, edgecolor='white', width=0.55)
for b, v in zip(bars, mae_vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.05,
            f'{v:.2f}d', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('MAE (days)', fontsize=11)
ax.set_title('MAE — All Models', fontsize=13, fontweight='bold', pad=10)
ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

# RMSE bar
ax = fig2.add_subplot(gs2[0, 2])
ax.set_facecolor('white')
bars = ax.bar(model_names, rmse_vals, color=bar_colors, edgecolor='white', width=0.55)
for b, v in zip(bars, rmse_vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.05,
            f'{v:.2f}d', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('RMSE (days)', fontsize=11)
ax.set_title('RMSE — All Models', fontsize=13, fontweight='bold', pad=10)
ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

# Actual vs Predicted (best model)
ax = fig2.add_subplot(gs2[1, 0:2])
ax.set_facecolor('white')
preds_b = best['preds']
lim_max = max(y_test.max(), preds_b.max()) * 1.02
ax.scatter(y_test, preds_b, alpha=0.25, s=12, color=C1, edgecolors='none',
           label='Predictions')
ax.plot([0, lim_max], [0, lim_max], color=C3, lw=2, ls='--',
        label='Perfect prediction')
ax.set_xlim(0, lim_max); ax.set_ylim(0, lim_max)
ax.set_xlabel('Actual LOS (days)', fontsize=11)
ax.set_ylabel('Predicted LOS (days)', fontsize=11)
ax.set_title(f'Actual vs Predicted — {best_name}\n'
             f'(R²={best["R2"]:.3f}, MAE={best["MAE"]:.2f} d)',
             fontsize=13, fontweight='bold', pad=10)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)

# Residual plot
ax = fig2.add_subplot(gs2[1, 2])
ax.set_facecolor('white')
residuals = y_test.values - preds_b
ax.scatter(preds_b, residuals, alpha=0.25, s=12, color=C4, edgecolors='none')
ax.axhline(0, color=C3, lw=2, ls='--')
ax.set_xlabel('Predicted LOS (days)', fontsize=11)
ax.set_ylabel('Residuals (days)', fontsize=11)
ax.set_title(f'Residual Plot — {best_name}', fontsize=13, fontweight='bold', pad=10)
ax.spines[['top', 'right']].set_visible(False)

fig2.suptitle('Hospital Length of Stay — Model Evaluation',
              fontsize=16, fontweight='bold', y=1.01, color='#1E293B')
path2 = os.path.join(OUTPUT_DIR, 'fig2_model_eval.png')
plt.savefig(path2, dpi=150, bbox_inches='tight', facecolor='#F8FAFC')
plt.close()
print(f"  Saved → {path2}")

# ── Figure 3 : Feature importance ────────────────────────────────────────────
# use a tree model regardless of which was "best" overall
fi_name  = best_name if best_name in {'Random Forest', 'Gradient Boosting'} \
           else 'Random Forest'
fi_model = results[fi_name]['model']

feat_imp = (pd.Series(fi_model.feature_importances_, index=X.columns)
              .sort_values(ascending=True)
              .tail(20))

fig3, ax = plt.subplots(figsize=(12, 8))
fig3.patch.set_facecolor('#F8FAFC')
ax.set_facecolor('white')
colors_fi = [C2 if v >= feat_imp.quantile(0.75) else C1 for v in feat_imp.values]
ax.barh(range(len(feat_imp)), feat_imp.values, color=colors_fi,
        edgecolor='white', height=0.7)
ax.set_yticks(range(len(feat_imp)))
ax.set_yticklabels(feat_imp.index, fontsize=9)
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title(f'Top 20 Feature Importances — {fi_name}',
             fontsize=14, fontweight='bold', pad=12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
path3 = os.path.join(OUTPUT_DIR, 'fig3_feature_importance.png')
plt.savefig(path3, dpi=150, bbox_inches='tight', facecolor='#F8FAFC')
plt.close()
print(f"  Saved → {path3}")

print("\n" + "=" * 60)
print("ALL DONE")
print("=" * 60)
print(f"Outputs written to: {OUTPUT_DIR}/")
print("  best_model.pkl          — trained model (pickle)")
print("  model_metrics.csv       — R², MAE, RMSE for all models")
print("  fig1_eda.png            — EDA plots")
print("  fig2_model_eval.png     — model comparison & residuals")
print("  fig3_feature_importance.png — feature importances")


STEP 1 — LOAD & EXPLORE
Raw shape          : (10000, 51)

Target column stats (minutes):
count    1.000000e+04
mean     1.379236e+04
std      6.244166e+04
min      1.435000e+03
25%      5.755000e+03
50%      8.635000e+03
75%      1.439500e+04
max      5.378395e+06
Name: hospital_los, dtype: float64

Missing values (% per column, top 15):
intraop_range_stii                99.6
intraop_mean_uo                   72.1
intraop_mean_abs_delta_uo         72.1
intraop_max_uo                    72.1
intraop_min_art_dbp               65.0
intraop_max_art_dbp               65.0
intraop_range_art_dbp             65.0
intraop_mean_abs_delta_art_sbp    65.0
preop_fibrinogen                  64.7
preop_bun                         55.0
preop_alp                         54.4
preop_total_protein               54.3
preop_albumin                     54.3
preop_hb                          51.7
preop_hct                         51.5
dtype: float64

Data types:
float64    47
int64       4
Name: count, dtype: